## 1.查询保险产品及其条款文件

In [1]:
from app.modules.product.service import ProductService
from app.infra.database import AsyncSessionFactory

async with AsyncSessionFactory() as session:
    #创建service
    service = ProductService(session)
    #查询
    products = await service.list_products()

## 2.加载、解析条款文档

In [3]:
product = products[0]
print(product)

Product({'name': '达尔文宝贝计划12号', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'insurer': '信美人寿相互保险社', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-11-12 09:03:06', 'category': 'critical_illness', 'id': 1, 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-11-12 09:03:06'})


In [4]:
from mineru import MinerU
from app.core.config import settings

client = MinerU(settings.rag.mineru_token)
result = client.extract(f"../../data/raw/kb/{product.clause_name}")


In [11]:
# 三、文档优化
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-v4.1-flash-1017",
    api_key=settings.llm.api_key,
    base_url=settings.llm.base_url,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=100000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])
markdown = response.content

## 文档切分

- 优先采用Markdown结构感知切分，得到父块（parent_chunk）
- 接下来使用递归切分，处理父块，得到子块，每个父块至少要有1个子块（因为向量库只需要子块）
  - 如果发现父块太大（超过chunk_size），会被递归切分，得到子块（child_chunk）
  - 如果发现父块大小合适，则直接冗余一份父块，作为子块（child_chunk）